# 0. Data Preparation

| 개념 | 의미 | 식당 체인 비유 |
| :--- | :--- | :--- |
| site | 물리적인 데이터센터나 지역 | 지점이 있는 도시 |
| cluster | GPU 서버 묶음. 하나의 스케줄러가 관리 | 그 도시의 주방 |
| base | 원본 기반 모델 (예: Qwen-7B 같은 공개 모델) | 원조 레시피 |
| model | 실제로 배포되는 모델. base를 그대로 쓰거나(b_) 파인튜닝한 변형(s_) | 레시피를 변형한 메뉴 |
| app | 그 모델을 서비스하는 하나의 배포 단위. 고객이 호출하는 엔드포인트 | 메뉴를 파는 매장 |
| instance | app을 실제로 돌리는 실행 복제본 하나. GPU를 할당받아 요청을 처리 | 매장에서 일하는 요리사 한 명 |

In [1]:
# 전체 데이터: instance_daily 61개 샤드를 모두 읽는다 (원본: part_0 하나만 HTTP Range로 추출)
# - 문자열로 읽은 뒤 category로 바꾸면 문자열 상태로 ~100GB가 먼저 올라가므로,
#   category로 쓸 11개 열은 처음부터 dictionary(=category)로 읽는다 (~20GB)
# - instance_id는 이후 어디에서도 쓰지 않으므로(넣으면 안 되는 것 목록) 읽지 않는다
# - category 순서는 원본 astype("category")와 같게 정렬해 둔다
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA_DIR = Path("clusterdata/cluster-trace-v2026-maas/data")
files = sorted((DATA_DIR / "instance_daily").glob("part_*.parquet"))
cat_cols = ["app_id", "model_id", "base_id", "cluster_id", "site_id", "gpu_type",
            "workload", "model_arch", "pd_role", "base_arch", "variant_arch"]
read_cols = [c for c in pq.read_schema(files[0]).names if c != "instance_id"]

table = pa.concat_tables([pq.read_table(f, columns=read_cols, read_dictionary=cat_cols) for f in files])
table = table.unify_dictionaries()
df = table.to_pandas(self_destruct=True)
del table
for c in cat_cols:
    df[c] = df[c].cat.reorder_categories(sorted(df[c].cat.categories))
print(f"샤드 {len(files)}개 -> df {df.shape}, day {df['day'].min()}–{df['day'].max()}")

샤드 61개 -> df (145083117, 22), day 0–181


# 1. Data Preprocessing

In [2]:
# 원본은 여기서 문자열 -> category 변환을 했지만 위에서 이미 category로 읽었다
mem_before = mem_after = df.memory_usage(deep=True).sum()

about 1.5 GB per parquet - By using category type -> about 0.43 GB

| Column | Unique Value Count | Before → After Conversion |
| :--- | :--- | :--- |
| instance_id | 2,332,023 | Kept as is (conversion increases size: 191MB → 199MB) |
| app_id, model_id | Around 20,000 | Approx. 175MB → 6MB |
| base_id, cluster_id, site_id, gpu_type | 29~697 | Approx. 180MB → 2~5MB |
| Remaining 5 columns (workload, pd_role, etc.) | 2~3 | Approx. 30MB → 2.4MB |

In [3]:
mem_after = df.memory_usage(deep=True).sum()
print(f"shape: {df.shape}")
print(f"category 변환: {cat_cols}")
print(f"메모리: {mem_before / 1e9:.2f} GB -> {mem_after / 1e9:.2f} GB")
df.info(memory_usage="deep")
df.head()

shape: (145083117, 22)
category 변환: ['app_id', 'model_id', 'base_id', 'cluster_id', 'site_id', 'gpu_type', 'workload', 'model_arch', 'pd_role', 'base_arch', 'variant_arch']
메모리: 15.44 GB -> 15.44 GB
<class 'pandas.DataFrame'>
RangeIndex: 145083117 entries, 0 to 145083116
Data columns (total 22 columns):
 #   Column         Dtype   
---  ------         -----   
 0   day            int64   
 1   app_id         category
 2   model_id       category
 3   base_id        category
 4   cluster_id     category
 5   site_id        category
 6   gpu_type       category
 7   workload       category
 8   model_arch     category
 9   params_b       float64 
 10  pd_role        category
 11  gpu_count      float64 
 12  gpu_hours      float64 
 13  samples        int64   
 14  zero_samples   int64   
 15  sm_util        float64 
 16  gpu_mem_util   float64 
 17  cpu_usage      float64 
 18  mem_usage      float64 
 19  base_arch      category
 20  variant_arch   category
 21  gpu_mem_alloc  float64 

,day,app_id,model_id,base_id,cluster_id,site_id,gpu_type,workload,model_arch,params_b,...,gpu_hours,samples,zero_samples,sm_util,gpu_mem_util,cpu_usage,mem_usage,base_arch,variant_arch,gpu_mem_alloc
0,41,app_2913620dfa26fdd9783c6731a8602c8cc811eaf527...,b_d13787c7256a9af8c4e7959bbfe130c150820283e132...,base_92fb6fb7f3c4c864725df5a0f421e0e1caf4aa4df...,cluster_e01391bcafe69718ac30789d211118bb79b5de...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,gpu_6e395263687cdf4bbd37a4c02e74f71da8df094a23...,offline,dense,72.0,...,0.127778,23,15,0.000773,2.427924,0.429500,0.621491,dense,dense,95.577148
1,41,app_2913620dfa26fdd9783c6731a8602c8cc811eaf527...,b_d13787c7256a9af8c4e7959bbfe130c150820283e132...,base_92fb6fb7f3c4c864725df5a0f421e0e1caf4aa4df...,cluster_15b6c65d2b6be2c2096c60f96fde365ae5fb24...,site_8f1ad249c2b70b80298d1fe6b9f79ec441c3ff223...,gpu_6e395263687cdf4bbd37a4c02e74f71da8df094a23...,offline,dense,72.0,...,0.127778,23,18,0.001499,2.452366,0.457528,0.761123,dense,dense,95.577148
2,41,app_2913620dfa26fdd9783c6731a8602c8cc811eaf527...,b_d13787c7256a9af8c4e7959bbfe130c150820283e132...,base_92fb6fb7f3c4c864725df5a0f421e0e1caf4aa4df...,cluster_15b6c65d2b6be2c2096c60f96fde365ae5fb24...,site_8f1ad249c2b70b80298d1fe6b9f79ec441c3ff223...,gpu_6e395263687cdf4bbd37a4c02e74f71da8df094a23...,offline,dense,72.0,...,0.127778,23,18,0.001245,2.645864,0.472386,0.850161,dense,dense,95.577148
3,41,app_2913620dfa26fdd9783c6731a8602c8cc811eaf527...,b_d13787c7256a9af8c4e7959bbfe130c150820283e132...,base_92fb6fb7f3c4c864725df5a0f421e0e1caf4aa4df...,cluster_e01391bcafe69718ac30789d211118bb79b5de...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,gpu_6e395263687cdf4bbd37a4c02e74f71da8df094a23...,offline,dense,72.0,...,0.122222,22,20,0.001375,3.125898,0.426525,0.799866,dense,dense,95.577148
4,41,app_2913620dfa26fdd9783c6731a8602c8cc811eaf527...,b_d13787c7256a9af8c4e7959bbfe130c150820283e132...,base_92fb6fb7f3c4c864725df5a0f421e0e1caf4aa4df...,cluster_15b6c65d2b6be2c2096c60f96fde365ae5fb24...,site_8f1ad249c2b70b80298d1fe6b9f79ec441c3ff223...,gpu_6e395263687cdf4bbd37a4c02e74f71da8df094a23...,offline,dense,72.0,...,0.127778,23,17,0.001512,2.780508,0.445806,0.715164,dense,dense,95.577148


### 1-1. EDA

In [4]:
# 결측치: 컬럼별 개수와 비율(%) (결측이 있는 컬럼만, 많은 순)
na = pd.DataFrame({"n_missing": df.isna().sum(), "pct_missing": (df.isna().mean() * 100).round(2)})
na = na[na["n_missing"] > 0].sort_values("n_missing", ascending=False)
print(f"결측치가 있는 컬럼: {len(na)} / {df.shape[1]}")
display(na)

# 수치형 컬럼 기본 통계
display(df.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(4))

# category 컬럼: 고유값 수, 최빈값과 그 비율(%)
cat = df.select_dtypes(include="category")
display(pd.DataFrame({
    "nunique": cat.nunique(),
    "top": cat.mode().iloc[0],
    "top_pct": cat.apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100).round(2),
}))

결측치가 있는 컬럼: 6 / 22


,n_missing,pct_missing
sm_util,32331290,22.28
params_b,17651959,12.17
gpu_mem_util,7147044,4.93
cpu_usage,5629076,3.88
mem_usage,1695750,1.17
gpu_mem_alloc,23,0.00


,count,mean,std,min,1%,25%,50%,75%,99%,max
day,145083117.0,84.3202,49.2125,0.0000,1.0000,41.0000,82.0000,126.0000,179.0000,181.0000
params_b,127431158.0,26.0341,60.8493,0.0500,0.6000,3.0000,7.0000,30.0000,235.0000,1050.0000
gpu_count,145083117.0,1.2300,0.8077,0.5000,1.0000,1.0000,1.0000,1.0000,4.0000,16.0000
gpu_hours,145083117.0,2.5216,9.1387,0.0056,0.0056,0.1167,0.3278,1.2056,27.8667,383.8222
samples,145083117.0,335.6199,851.6050,1.0000,1.0000,19.0000,51.0000,195.0000,4317.0000,4320.0000
zero_samples,145083117.0,73.3432,386.7193,0.0000,0.0000,5.0000,12.0000,23.0000,1936.0000,4320.0000
sm_util,112751827.0,25.9627,28.3470,0.0000,0.0000,0.0676,15.6441,47.4009,90.1867,13280.6009
gpu_mem_util,137936073.0,63.2402,37.0844,0.0000,0.0000,28.8704,82.1234,92.1202,99.1986,298.5589
cpu_usage,139454041.0,1.1945,1.3916,0.0000,0.0128,0.4880,0.9890,1.4228,7.5477,128.8940
mem_usage,143387367.0,9.2677,16.0104,-18455.4539,0.0000,2.5530,5.4931,11.4899,74.7098,951.9657


,nunique,top,top_pct
app_id,298142,app_ae11f02a011633a0cec9d63077fbf8e8c267d23f92...,1.13
model_id,232180,o_c92dcf326ef4268c49a99ea397fec7807b1b117b9295...,5.52
base_id,1324,base_97dadfec33b567a8c46f553ecc9c038b2b1656b11...,6.69
cluster_id,56,cluster_e01391bcafe69718ac30789d211118bb79b5de...,9.22
site_id,61,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,20.75
gpu_type,32,gpu_d7bba4af71a4b63f56fa71f941b73d04f0a776f674...,15.18
workload,2,offline,95.86
model_arch,2,dense,87.99
pd_role,3,fused,99.83
base_arch,3,dense,70.11


### 1.2 Preprocessing

입력: gpu_type, gpu_count, gpu_mem_alloc, workload, pd_role, params_b, params_b_is_missing, model_arch, base_arch, variant_arch, model_source, cluster_id, site_id, day_of_week

타겟: idle_ratio, is_idle

넣으면 안되는 것: sm_util, gpu_mem_util, cpu_usage, mem_usage, samples, zero_samples, gpu_hours, instance_id, app_id, model_id, base_id

In [5]:
# 전처리용 scikit-learn 호환 transformer
# 모두 학습할 상태가 없는(stateless) 변환이며, 입력 DataFrame은 수정하지 않고 복사본을 반환한다.
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


class IdleRatioTransformer(BaseEstimator, TransformerMixin):
    """
    zero_samples와 samples를 이용하여 idle_ratio(회귀)를 계산하고,
    특정 임계값 이상인지에 대한 플래그(분류)를 생성합니다.

    - 회귀: idle_ratio = zero_samples / samples (관측 중 GPU가 놀고 있던 비율)
    - 분류: idle_ratio >= threshold (기본 0.9) 이면 1 ("사실상 놀기만 한 세그먼트"), 그 외 0
    """

    def __init__(self,
                 zero_samples_col="zero_samples",
                 samples_col="samples",
                 threshold=0.9,
                 ratio_col="idle_ratio",
                 flag_col="is_mostly_idle"):

        self.zero_samples_col = zero_samples_col
        self.samples_col = samples_col
        self.threshold = threshold
        self.ratio_col = ratio_col
        self.flag_col = flag_col

    def fit(self, X, y=None):
        # 학습할 파라미터가 없으므로 self만 반환합니다.
        return self

    def transform(self, X):
        X = X.copy()

        zero_samples = X[self.zero_samples_col]
        samples = X[self.samples_col]

        # 1. 회귀: idle_ratio 계산
        # 분모(samples)가 0인 경우 0으로 나누는 에러 방지를 위해 np.where 사용 (0일 경우 NaN 처리)
        X[self.ratio_col] = np.where(
            samples > 0,
            zero_samples / samples,
            np.nan
        )

        # 2. 분류: idle_ratio >= 0.9 이면 1, 아니면 0
        X[self.flag_col] = (X[self.ratio_col] >= self.threshold).astype("int8")

        return X


class NegativeValueCleaner(BaseEstimator, TransformerMixin):
    """음수 값 정제 + outlier 플래그.

    - noise_threshold < 값 < 0 : 미세 음수(뺄셈/반올림 노이즈) -> 0, 플래그 0
    - 값 <= noise_threshold     : 크게 벗어난 음수(수집 오류)    -> NaN, 플래그 1
    - 그 외                     : 그대로, 플래그 0

    mem_usage(GiB) 기준 기본값 -0.01은 10 MiB 미만을 노이즈로 본다.
    플래그 컬럼 이름은 기본 `outlier_{column}`.
    """

    def __init__(self, column="mem_usage", noise_threshold=-0.01, flag_column=None):
        self.column = column
        self.noise_threshold = noise_threshold
        self.flag_column = flag_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        s = X[self.column]
        tiny = (s < 0) & (s > self.noise_threshold)
        outlier = s <= self.noise_threshold

        X.loc[tiny, self.column] = 0.0
        X.loc[outlier, self.column] = np.nan
        X[self.flag_column or f"outlier_{self.column}"] = outlier.astype("int8")
        return X


class UpperBoundCleaner(BaseEstimator, TransformerMixin):
    """상한 초과 값 정제 + outlier 플래그.

    - 값 > upper : NaN, 플래그 1
    - 그 외      : 그대로, 플래그 0 (원래 NaN인 행도 0)

    플래그 컬럼 이름은 기본 `outlier_{column}`.
    """

    def __init__(self, column="sm_util", upper=100.0, flag_column=None):
        self.column = column
        self.upper = upper
        self.flag_column = flag_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        outlier = X[self.column] > self.upper

        X.loc[outlier, self.column] = np.nan
        X[self.flag_column or f"outlier_{self.column}"] = outlier.astype("int8")
        return X


class AppDayAggregator(BaseEstimator, TransformerMixin):
    """instance 세그먼트 행들을 (app_id, day)당 한 행으로 집계한다.

    - sum : samples, zero_samples (-> app-day idle_ratio 재계산용), gpu_hours (다음 날 값이 가중치가 됨)
    - mean: 수치형 변수 (NaN 제외 단순 평균)
    - max : outlier_* 플래그 (그날 하나라도 이상값이면 1)
    - mode: 범주형 변수 (그날 가장 많이 등장한 값)
    """

    def __init__(self,
                 group_cols=("app_id", "day"),
                 sum_cols=("samples", "zero_samples", "gpu_hours"),
                 mean_cols=("gpu_count", "gpu_mem_alloc", "params_b",
                            "mem_usage", "sm_util", "gpu_mem_util"),
                 max_cols=("outlier_mem_usage", "outlier_sm_util", "outlier_gpu_mem_util"),
                 mode_cols=("gpu_type", "workload", "pd_role",
                            "model_arch", "base_arch", "variant_arch",
                            "cluster_id", "site_id")):
        self.group_cols = group_cols
        self.sum_cols = sum_cols
        self.mean_cols = mean_cols
        self.max_cols = max_cols
        self.mode_cols = mode_cols

    def fit(self, X, y=None):
        return self

    @staticmethod
    def _mode(X, keys, col):
        # 그룹별 최빈값: (keys, col) 조합 개수를 세서 그룹마다 가장 많은 값 하나만 남김
        counts = X.groupby(keys + [col], observed=True).size().rename("n").reset_index()
        counts = counts.sort_values("n", ascending=False, kind="stable").drop_duplicates(keys)
        return counts.set_index(keys)[col]

    def transform(self, X):
        keys = list(self.group_cols)
        g = X.groupby(keys, observed=True, sort=True)
        out = pd.concat([
            g[list(self.sum_cols)].sum(),
            g[list(self.mean_cols)].mean(),
            g[list(self.max_cols)].max(),
        ], axis=1)
        for c in self.mode_cols:
            out[c] = self._mode(X, keys, c)
        return out.reset_index()


class NextDayTargetShifter(BaseEstimator, TransformerMixin):
    """같은 app의 day + 1 타겟을 오늘(day) 행에 `{target}{suffix}`로 붙인다.

    - day + 1 기록만 인정 (며칠 뒤 다음 등장은 사용하지 않음)
    - dropna=True면 day + 1 기록이 없는 행은 제거
    - 타겟 is_mostly_idle과 가중치 gpu_hours를 함께 민다.
      (예측하는 것도, 회수할 수 있는 GPU도 다음 날 것이므로 가중치도 다음 날 gpu_hours여야 함)
    - idle_ratio는 is_mostly_idle 계산에만 쓰이는 중간값이라 밀지 않는다.
    """

    def __init__(self, group_col="app_id", day_col="day",
                 target_cols=("is_mostly_idle", "gpu_hours"),
                 suffix="_next", dropna=True):
        self.group_col = group_col
        self.day_col = day_col
        self.target_cols = target_cols
        self.suffix = suffix
        self.dropna = dropna

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        g, d, targets = self.group_col, self.day_col, list(self.target_cols)
        new_cols = [c + self.suffix for c in targets]

        # (app, day + 1)의 타겟을 (app, day) 키로 옮겨서 붙임
        nxt = X[[g, d] + targets].copy()
        nxt[d] = nxt[d] - 1
        nxt = nxt.rename(columns=dict(zip(targets, new_cols)))
        out = X.merge(nxt, on=[g, d], how="left", validate="one_to_one")

        if self.dropna:
            out = out.dropna(subset=new_cols).reset_index(drop=True)
            for c, n in zip(targets, new_cols):
                out[n] = out[n].astype(X[c].dtype)
        return out


class SelectVariables(BaseEstimator, TransformerMixin):
    """input 변수와 타겟, 보조 변수만 남기고 나머지(넣으면 안 되는 것)는 모두 제거한다.

    - input: 실행 전에 알 수 있는 정보만 (파생 변수 params_b_is_missing, model_source,
             day_of_week는 아직 만들지 않았으므로 제외)
    - target: IdleRatioTransformer가 만든 idle_ratio, is_mostly_idle
    - 제거: sm_util, gpu_mem_util, cpu_usage, mem_usage, samples, zero_samples, gpu_hours,
            instance_id, app_id, model_id, base_id, day, outlier_* 플래그
    """

    def __init__(self,
                 input_cols=("gpu_type", "gpu_count", "gpu_mem_alloc",
                             "gpu_hours",  # 오늘(day)의 GPU-hours 합: 하루가 끝나면 아는 값이라 다음 날 예측에 써도 누수 아님
                             "workload", "pd_role",
                             "params_b", "model_arch", "base_arch", "variant_arch",
                             "cluster_id", "site_id",
                             "mem_usage", "outlier_mem_usage",
                             "sm_util", "outlier_sm_util",
                             "gpu_mem_util", "outlier_gpu_mem_util",),
                 target_cols=("idle_ratio", "is_mostly_idle"),
                 auxiliary_cols=("day", "gpu_hours")):
        self.input_cols = input_cols
        self.target_cols = target_cols
        self.auxiliary_cols = auxiliary_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cols = list(self.input_cols) + list(self.target_cols) + list(self.auxiliary_cols)
        missing = [c for c in cols if c not in X.columns]
        if missing:
            raise KeyError(f"SelectInputVariables: 입력에 없는 컬럼 {missing}")
        return X[cols].copy()


def build_preprocessing_pipeline(idle_threshold=0.9):
    """1. 정제 -> 2. app x day 집계 -> 3. 타겟 생성 -> 4. 타겟 day + 1로 밀기 -> 5. 변수 선택."""
    return Pipeline([
        ("mem_usage_negative", NegativeValueCleaner("mem_usage", noise_threshold=-0.01)),
        ("sm_util_over_100", UpperBoundCleaner("sm_util", upper=100.0)),
        ("gpu_mem_util_over_100", UpperBoundCleaner("gpu_mem_util", upper=100.0)),
        ("app_day_aggregate", AppDayAggregator()),
        ("create_target_vars", IdleRatioTransformer(threshold=idle_threshold)),
        ("shift_target_next_day", NextDayTargetShifter()),
        ("select_vars", SelectVariables(target_cols=("is_mostly_idle_next",),
                                        auxiliary_cols=("day", "gpu_hours_next"))),
    ])

근데 화면공유 꺼야할거 같아요

In [6]:
# 전처리 파이프라인 실행: 원본 df는 그대로 두고 결과를 preprocessed_df에 저장
# 1) 정제
#   - mem_usage:    (-0.01, 0) 미세 음수 -> 0 / <= -0.01 -> NaN + outlier_mem_usage = 1
#   - sm_util:      > 100 -> NaN + outlier_sm_util = 1
#   - gpu_mem_util: > 100 -> NaN + outlier_gpu_mem_util = 1
# 2) app x day 집계: instance 세그먼트 행 -> (app_id, day)당 한 행
# 3) 타겟 생성: app-day 단위 idle_ratio -> is_mostly_idle (idle_ratio >= IDLE_THRESHOLD)
# 4) 타겟 밀기: 같은 app의 day + 1 값 -> is_mostly_idle_next, gpu_hours_next (day + 1 없는 행 제거)
# 5) input + 타겟(is_mostly_idle_next) + 보조 변수(day, gpu_hours_next)만 남김

# 정답 threshold: idle_ratio >= IDLE_THRESHOLD 이면 정답을 idle(1)로 본다
# 후보: np.linspace(0.85, 0.95, 11) -> 0.85, 0.86, ..., 0.95 중 하나를 직접 넣어가며 비교
# (주의: 값이 바뀌면 idle(1) 비율도 바뀌므로, 점수 크기만으로 기준끼리 우열을 가리지 않는다)
IDLE_THRESHOLD = 0.9
preprocessed_df = build_preprocessing_pipeline(idle_threshold=IDLE_THRESHOLD).fit_transform(df)

print(f"df: {df.shape} -> preprocessed_df: {preprocessed_df.shape}")
print(f"컬럼: {list(preprocessed_df.columns)}")
print(f"IDLE_THRESHOLD = {IDLE_THRESHOLD} -> is_mostly_idle_next = 1 비율: {preprocessed_df['is_mostly_idle_next'].mean() * 100:.2f}%")
print(f"결측이 있는 컬럼: {preprocessed_df.isna().sum()[lambda s: s > 0].to_dict()}")

df: (145083117, 22) -> preprocessed_df: (535846, 21)
컬럼: ['gpu_type', 'gpu_count', 'gpu_mem_alloc', 'gpu_hours', 'workload', 'pd_role', 'params_b', 'model_arch', 'base_arch', 'variant_arch', 'cluster_id', 'site_id', 'mem_usage', 'outlier_mem_usage', 'sm_util', 'outlier_sm_util', 'gpu_mem_util', 'outlier_gpu_mem_util', 'is_mostly_idle_next', 'day', 'gpu_hours_next']
IDLE_THRESHOLD = 0.9 -> is_mostly_idle_next = 1 비율: 35.70%
결측이 있는 컬럼: {'params_b': 75571, 'mem_usage': 7242, 'sm_util': 48118, 'gpu_mem_util': 7283}


In [7]:
preprocessed_df.columns

Index(['gpu_type', 'gpu_count', 'gpu_mem_alloc', 'gpu_hours', 'workload',
       'pd_role', 'params_b', 'model_arch', 'base_arch', 'variant_arch',
       'cluster_id', 'site_id', 'mem_usage', 'outlier_mem_usage', 'sm_util',
       'outlier_sm_util', 'gpu_mem_util', 'outlier_gpu_mem_util',
       'is_mostly_idle_next', 'day', 'gpu_hours_next'],
      dtype='str')

In [8]:
preprocessed_df.head()

,gpu_type,gpu_count,gpu_mem_alloc,gpu_hours,workload,pd_role,params_b,model_arch,base_arch,variant_arch,...,site_id,mem_usage,outlier_mem_usage,sm_util,outlier_sm_util,gpu_mem_util,outlier_gpu_mem_util,is_mostly_idle_next,day,gpu_hours_next
0,gpu_a4c9b5cf4ae29324db1b9844d6742afa8a7b7ab76a...,1.0,123.188133,324.200000,online,fused,NaN,dense,dense,dense,...,site_e3b33cae10ea8c15c1160784b075b9690f05dde81...,6.415385,0,0.000000,0,97.794382,0,0,42,866.366667
1,gpu_a4c9b5cf4ae29324db1b9844d6742afa8a7b7ab76a...,1.0,130.280134,866.366667,online,fused,NaN,dense,dense,dense,...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,7.479337,0,4.814130,0,98.288336,0,0,43,958.611111
2,gpu_a4c9b5cf4ae29324db1b9844d6742afa8a7b7ab76a...,1.0,141.706845,958.611111,online,fused,NaN,dense,dense,dense,...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,8.342272,0,10.886934,0,98.704638,0,0,44,918.477778
3,gpu_a4c9b5cf4ae29324db1b9844d6742afa8a7b7ab76a...,1.0,141.706845,918.477778,online,fused,NaN,dense,dense,dense,...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,8.403995,0,9.849198,0,98.718950,0,0,45,1254.666667
4,gpu_a4c9b5cf4ae29324db1b9844d6742afa8a7b7ab76a...,1.0,143.992188,1254.666667,online,fused,NaN,dense,dense,dense,...,site_941df8d86fd9aeb89fe94d58968d015bad19dee55...,8.638541,0,8.767994,0,98.773230,0,0,46,1916.922222


# 2. Construct Various Models

In [9]:
# 모델별 인코딩 파이프라인
# 1.2 전처리와 달리 인코더는 "어떤 범주가 있는지"를 데이터에서 학습(fit)한다.
# -> 반드시 train으로만 fit하고 val / test에는 transform만 한다. (모델 파이프라인 안에 넣어서 사용)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

TARGET_COLS = ["is_mostly_idle_next"]
AUX_COLS = ["day", "gpu_hours_next"]


def split_xyw(data, target="is_mostly_idle_next"):
    """preprocessed_df -> (X, y, w). X에는 타겟과 보조 변수가 들어가지 않는다.

    - y: is_mostly_idle_next (다음 날 사실상 놀기만 할지, 0/1)
    - w: gpu_hours_next (sample_weight) - 다음 날 회수할 수 있는 GPU 양
    """
    X = data.drop(columns=TARGET_COLS + AUX_COLS)
    # .copy(): pandas 3에서 이 열은 2D 블록의 strided view라 SVC(희소 입력)가 C-contiguous 아님 에러를 냄
    return X, data[target], data["gpu_hours_next"].copy()


class ColumnsByDtype:
    """ColumnTransformer용 컬럼 선택기: dtype("category" / "number")으로 고르고 exclude는 뺀다.

    - exclude: 뺄 컬럼 이름
    - exclude_prefix: 이 접두어로 시작하는 컬럼도 뺀다 (예: "outlier_")
    (속성 이름을 dtype으로 하면 sklearn이 배열로 오인하므로 include를 쓴다)
    """

    def __init__(self, include, exclude=(), exclude_prefix=()):
        self.include = include
        self.exclude = tuple(exclude)
        self.exclude_prefix = tuple(exclude_prefix)

    def __call__(self, X):
        return [c for c in X.select_dtypes(include=self.include).columns
                if c not in self.exclude and not c.startswith(self.exclude_prefix)]


class ColumnsByPrefix:
    """ColumnTransformer용 컬럼 선택기: 이름이 prefix로 시작하는 컬럼을 고른다."""

    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, X):
        return [c for c in X.columns if c.startswith(self.prefix)]


# ---------------------------------------------------------------------------
# Logistic regression / SVM용
# ---------------------------------------------------------------------------
def build_linear_encoder(min_frequency=0.01, exclude=("site_id",), flag_prefix="outlier_",
                         log_cols=("gpu_hours",)):
    """범주형 one-hot + 수치형 결측 대체·표준화 + 플래그는 0/1 그대로.

    - 범주형: OneHotEncoder
        * min_frequency: 전체의 1% 미만인 드문 범주는 "infrequent" 컬럼 하나로 묶음
        * handle_unknown: train에 없던 범주가 val / test에 나오면 infrequent로 처리 (에러 방지)
        * NaN은 하나의 범주로 취급
    - 수치형: 중앙값으로 결측 대체 -> StandardScaler (선형 모델은 NaN과 스케일 차이에 민감)
    - log_cols(gpu_hours): 중앙값 대체 -> log1p -> StandardScaler
        (0.03 ~ 수만까지 치우친 분포라 표준화만으로는 극단값에 계수가 끌려감)
    - 플래그(outlier_*): 0/1 그대로 통과 (드문 0/1 값을 표준화하면 1이 7~19 같은 큰 값이 되므로)
    - exclude: cluster_id와 거의 1:1인 site_id는 중복 정보라 기본으로 제외
    """
    categorical = OneHotEncoder(min_frequency=min_frequency, handle_unknown="infrequent_if_exist")
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    log_numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("cat", categorical, ColumnsByDtype("category", exclude)),
        ("num", numeric, ColumnsByDtype("number", tuple(exclude) + tuple(log_cols),
                                        exclude_prefix=(flag_prefix,))),
        ("log", log_numeric, list(log_cols)),
        ("flag", "passthrough", ColumnsByPrefix(flag_prefix)),
    ], remainder="drop")


# ---------------------------------------------------------------------------
# LightGBM / XGBoost용
# ---------------------------------------------------------------------------
class CategoryAligner(BaseEstimator, TransformerMixin):
    """one-hot 없이 category dtype 그대로 넘긴다 (트리 모델이 범주형을 직접 처리).

    fit 때 train에 실제로 등장한 범주만 기억하고, transform 때 범주 목록을 그것으로 맞춘다.
    -> train / val / test의 범주 코드가 항상 같고, train에 없던 범주는 NaN(결측)으로 처리된다.
    수치형 NaN은 트리 모델이 직접 처리하므로 건드리지 않는다.
    """

    def __init__(self, exclude=()):
        self.exclude = exclude

    def fit(self, X, y=None):
        cols = ColumnsByDtype("category", self.exclude)(X)
        self.categories_ = {c: X[c].cat.remove_unused_categories().cat.categories for c in cols}
        return self

    def transform(self, X):
        X = X.drop(columns=[c for c in self.exclude if c in X.columns]).copy()
        for c, cats in self.categories_.items():
            X[c] = X[c].cat.set_categories(cats)
        return X


def build_tree_encoder(exclude=()):
    """트리 모델은 cluster_id와 site_id의 중복에 크게 영향받지 않으므로 기본으로 둘 다 사용."""
    return CategoryAligner(exclude=exclude)


# ---------------------------------------------------------------------------
# 사용 예 (모델 셀에서)
#   X_train, y_train, w_train = split_xyw(train_df)
#   model = Pipeline([("encode", build_linear_encoder()),
#                     ("model", LogisticRegression(max_iter=1000))])
#   model.fit(X_train, y_train, model__sample_weight=w_train)
# ---------------------------------------------------------------------------

# 동작 확인 (테스트용: 분할 없이 preprocessed_df 전체로 fit)
X, y, w = split_xyw(preprocessed_df)
X_linear = build_linear_encoder().fit_transform(X)
X_tree = build_tree_encoder().fit_transform(X)
print(f"X: {X.shape} -> linear: {X_linear.shape}, tree: {X_tree.shape}")
print(f"tree 범주형 컬럼: {list(X_tree.select_dtypes(include='category').columns)}")

X: (535846, 18) -> linear: (535846, 60), tree: (535846, 18)
tree 범주형 컬럼: ['gpu_type', 'workload', 'pd_role', 'model_arch', 'base_arch', 'variant_arch', 'cluster_id', 'site_id']


In [10]:
# 데이터 분할 + 그리드서치 공통
# - 분할: Train 0–132 / Val 133–139 / Test 140–146 (시간 순)
# - 그리드서치: Train으로 학습, Val로 채점하는 1회 분할 (PredefinedSplit)
# - gpu_hours_next(다음 날 GPU-hours) 가중치는 모델 학습과 채점 모두에 반영 (sklearn metadata routing)
import inspect

from sklearn import clone, config_context
from sklearn.metrics import get_scorer
from sklearn.model_selection import GridSearchCV, PredefinedSplit, StratifiedKFold

TRAIN_DAYS, VAL_DAYS, TEST_DAYS = (0, 132), (133, 139), (140, 146)


def in_days(data, days):
    return data["day"].between(*days)


train_val_df = preprocessed_df[in_days(preprocessed_df, (TRAIN_DAYS[0], VAL_DAYS[1]))]
test_df = preprocessed_df[in_days(preprocessed_df, TEST_DAYS)]
print(f"train+val: {len(train_val_df):,} rows | test: {len(test_df):,} rows")


def make_cv(data):
    """Train(0–132)으로 학습, Val(133–139)로 채점하는 1회 분할.

    Train 또는 Val 구간이 비어 있으면(테스트 환경) 3-fold로 대체한다.
    """
    is_val = in_days(data, VAL_DAYS).to_numpy()
    if is_val.any() and (~is_val).any():
        return PredefinedSplit(np.where(is_val, 0, -1)), "PredefinedSplit (train 0–132 / val 133–139)"
    return StratifiedKFold(3, shuffle=True, random_state=0), "StratifiedKFold(3) - val 구간 데이터 없음 (테스트 환경)"


def route_weight_to_model_only(pipeline):
    """sample_weight를 마지막 모델에만 전달하고, 인코더 안의 단계(StandardScaler 등)에는 전달하지 않는다."""
    for est in pipeline.get_params(deep=True).values():
        if hasattr(est, "set_fit_request") and "sample_weight" in inspect.signature(est.fit).parameters:
            est.set_fit_request(sample_weight=False)
    pipeline.steps[-1][1].set_fit_request(sample_weight=True)
    return pipeline


def run_grid_search(pipeline, param_grid, data, scoring="average_precision"):
    """gpu_hours_next 가중치를 학습과 채점 모두에 반영하는 GridSearchCV (타겟: is_mostly_idle_next).

    - 기본 scoring = average_precision (불균형 데이터에 적합한 PR-AUC)
    - 가장 좋은 파라미터로 train+val 전체에 다시 학습(refit)한 모델이 best_estimator_에 들어 있다.
    """
    X, y, w = split_xyw(data)
    cv, cv_name = make_cv(data)

    with config_context(enable_metadata_routing=True):
        pipe = route_weight_to_model_only(clone(pipeline))
        scorer = get_scorer(scoring).set_score_request(sample_weight=True)
        search = GridSearchCV(pipe, param_grid, scoring=scorer, cv=cv, n_jobs=-1)
        search.fit(X, y, sample_weight=w)

    print(f"CV: {cv_name} | rows: {len(X):,} | 조합 {len(search.cv_results_['params'])}개")
    print(f"best {scoring}: {search.best_score_:.4f}")
    print(f"best params: {search.best_params_}")
    top = (pd.DataFrame(search.cv_results_)
           .sort_values("rank_test_score")
           .loc[:, ["rank_test_score", "mean_test_score", "params"]]
           .head(5))
    display(top)
    return search


# ---------------------------------------------------------------------------
# 결정 threshold 튜닝 (for문)
# ---------------------------------------------------------------------------
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

THRESHOLDS = np.linspace(0.01, 0.99, 100)   # 결정 threshold 후보 (값을 직접 바꿔가며 사용)


def tune_decision_threshold(search, data, thresholds=THRESHOLDS):
    """결정 threshold(확률 몇 이상이면 idle로 판정할지)를 하나씩 바꿔가며 F1을 비교한다.

    1) train으로 모델 학습 (그리드서치에서 고른 파라미터 그대로)
    2) val에서 "내일 idle일 확률" 계산
    3) threshold마다 "확률 >= threshold 이면 idle" 로 판정 -> gpu_hours_next 가중 F1 계산
    4) F1이 가장 높은 threshold를 고른다
    """
    X, y, w = split_xyw(data)

    # train / val 나누기 (val 구간이 비어 있으면 테스트 환경이므로 무작위 7:3)
    is_val = in_days(data, VAL_DAYS)
    if is_val.any() and (~is_val).any():
        X_tr, y_tr, w_tr = X[~is_val], y[~is_val], w[~is_val]
        X_va, y_va, w_va = X[is_val], y[is_val], w[is_val]
    else:
        X_tr, X_va, y_tr, y_va, w_tr, w_va = train_test_split(
            X, y, w, test_size=0.3, stratify=y, random_state=0)

    model = clone(search.best_estimator_)
    model.fit(X_tr, y_tr, model__sample_weight=w_tr)
    prob = model.predict_proba(X_va)[:, 1]

    rows = []
    for t in thresholds:
        pred = (prob >= t).astype(int)
        f1 = f1_score(y_va, pred, sample_weight=w_va, zero_division=0)
        rows.append({"threshold": t, "F1": f1})
    table = pd.DataFrame(rows)

    best = table.loc[table["F1"].idxmax()]
    print(f"val rows: {len(X_va):,} | 기준: gpu_hours_next 가중 F1")
    print(f"best threshold: {best['threshold']:.3f} (F1 = {best['F1']:.4f})")
    return best["threshold"], table

train+val: 414,663 rows | test: 21,342 rows


### 2.1 Logistic regression

In [11]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("encode", build_linear_encoder()),
    ("model", LogisticRegression(max_iter=2000)),
])
lr_param_grid = {
    "model__C": [0.01, 0.1, 1, 10],               # 규제 강도의 역수 (작을수록 강한 규제)
    "model__class_weight": [None, "balanced"],    # 소수 클래스(idle) 가중
}
lr_search = run_grid_search(lr_pipeline, lr_param_grid, train_val_df)

# 결정 threshold 튜닝: THRESHOLDS 중 gpu_hours_next 가중 F1이 최대가 되는 확률
lr_threshold, lr_threshold_table = tune_decision_threshold(lr_search, train_val_df)

CV: PredefinedSplit (train 0–132 / val 133–139) | rows: 414,663 | 조합 8개
best average_precision: 0.6726
best params: {'model__C': 1, 'model__class_weight': None}


,rank_test_score,mean_test_score,params
4,1,0.672604,"{'model__C': 1, 'model__class_weight': None}"
6,2,0.672026,"{'model__C': 10, 'model__class_weight': None}"
0,3,0.671557,"{'model__C': 0.01, 'model__class_weight': None}"
2,4,0.671258,"{'model__C': 0.1, 'model__class_weight': None}"
3,5,0.608135,"{'model__C': 0.1, 'model__class_weight': 'bala..."


val rows: 20,835 | 기준: gpu_hours_next 가중 F1
best threshold: 0.446 (F1 = 0.6594)


### 2.2 SVM

In [12]:
from sklearn.svm import SVC

# 주의: SVC(RBF 커널)는 학습 시간이 행 수의 제곱 이상으로 늘어난다.
#       probability=True는 확률 보정을 위해 내부 CV를 한 번 더 해서 더 느리다.
#       전체 데이터에서 너무 느리면 LinearSVC 또는 샘플링을 고려.
svm_pipeline = Pipeline([
    ("encode", build_linear_encoder()),
    ("model", SVC(kernel="rbf", probability=True, random_state=0)),  # threshold 튜닝에 확률(0~1) 필요
])
svm_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.01, 0.1],
}
# 전체 데이터에서는 RBF SVC를 train 전체(~39만 행)로 학습할 수 없으므로 train만 표본 추출 (val은 전부)
SVM_TRAIN_N = 20_000
_is_val = in_days(train_val_df, VAL_DAYS)
svm_df = pd.concat([train_val_df[~_is_val].sample(n=SVM_TRAIN_N, random_state=0), train_val_df[_is_val]])
print(f"SVM용: train {SVM_TRAIN_N:,} 행 표본 + val {_is_val.sum():,} 행")
svm_search = run_grid_search(svm_pipeline, svm_param_grid, svm_df)

# 결정 threshold 튜닝: THRESHOLDS 중 gpu_hours_next 가중 F1이 최대가 되는 확률
svm_threshold, svm_threshold_table = tune_decision_threshold(svm_search, svm_df)

SVM용: train 20,000 행 표본 + val 20,835 행


/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was depreca

/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


CV: PredefinedSplit (train 0–132 / val 133–139) | rows: 40,835 | 조합 9개
best average_precision: 0.7578
best params: {'model__C': 10, 'model__gamma': 0.01}


,rank_test_score,mean_test_score,params
7,1,0.757788,"{'model__C': 10, 'model__gamma': 0.01}"
4,2,0.748799,"{'model__C': 1, 'model__gamma': 0.01}"
1,3,0.742269,"{'model__C': 0.1, 'model__gamma': 0.01}"
0,4,0.716502,"{'model__C': 0.1, 'model__gamma': 'scale'}"
3,5,0.714807,"{'model__C': 1, 'model__gamma': 'scale'}"


/ssd/jylee/gpu-reclaim/env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


val rows: 20,835 | 기준: gpu_hours_next 가중 F1
best threshold: 0.653 (F1 = 0.7239)


### 2.3 LightGBM

In [13]:
from lightgbm import LGBMClassifier

lgbm_pipeline = Pipeline([
    ("encode", build_tree_encoder()),   # category dtype 그대로 -> LightGBM이 범주형으로 직접 처리
    ("model", LGBMClassifier(random_state=0, n_jobs=1, verbose=-1)),
])
lgbm_param_grid = {
    "model__n_estimators": [100, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [15, 31],
}
lgbm_search = run_grid_search(lgbm_pipeline, lgbm_param_grid, train_val_df)

# 결정 threshold 튜닝: THRESHOLDS 중 gpu_hours_next 가중 F1이 최대가 되는 확률
lgbm_threshold, lgbm_threshold_table = tune_decision_threshold(lgbm_search, train_val_df)

CV: PredefinedSplit (train 0–132 / val 133–139) | rows: 414,663 | 조합 8개
best average_precision: 0.9002
best params: {'model__learning_rate': 0.1, 'model__n_estimators': 300, 'model__num_leaves': 31}


,rank_test_score,mean_test_score,params
7,1,0.900188,"{'model__learning_rate': 0.1, 'model__n_estima..."
3,2,0.892797,"{'model__learning_rate': 0.05, 'model__n_estim..."
6,3,0.890909,"{'model__learning_rate': 0.1, 'model__n_estima..."
5,4,0.875963,"{'model__learning_rate': 0.1, 'model__n_estima..."
2,5,0.875667,"{'model__learning_rate': 0.05, 'model__n_estim..."


val rows: 20,835 | 기준: gpu_hours_next 가중 F1
best threshold: 0.366 (F1 = 0.8293)


### 2.4 XGBoost

In [14]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("encode", build_tree_encoder()),   # category dtype 그대로 -> enable_categorical=True로 직접 처리
    ("model", XGBClassifier(enable_categorical=True, tree_method="hist",
                            random_state=0, n_jobs=1, eval_metric="logloss")),
])
xgb_param_grid = {
    "model__n_estimators": [100, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 6],
}
xgb_search = run_grid_search(xgb_pipeline, xgb_param_grid, train_val_df)

# 결정 threshold 튜닝: THRESHOLDS 중 gpu_hours_next 가중 F1이 최대가 되는 확률
xgb_threshold, xgb_threshold_table = tune_decision_threshold(xgb_search, train_val_df)

CV: PredefinedSplit (train 0–132 / val 133–139) | rows: 414,663 | 조합 8개
best average_precision: 0.8635
best params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 300}


,rank_test_score,mean_test_score,params
5,1,0.863510,"{'model__learning_rate': 0.1, 'model__max_dept..."
1,2,0.841064,"{'model__learning_rate': 0.05, 'model__max_dep..."
4,3,0.814829,"{'model__learning_rate': 0.1, 'model__max_dept..."
0,4,0.799581,"{'model__learning_rate': 0.05, 'model__max_dep..."
3,5,0.775990,"{'model__learning_rate': 0.05, 'model__max_dep..."


val rows: 20,835 | 기준: gpu_hours_next 가중 F1
best threshold: 0.267 (F1 = 0.7891)
